# Replikasi — Khan, Majumdar & Mondal (2025)
## *Sentiment analysis of emoji fused reviews using machine learning and BERT*

**Jurnal:** Scientific Reports 15:7538 · DOI `10.1038/s41598-025-92286-0`

---

### Status verifikasi (baca dulu)

| Komponen | Status |
|---|---|
| Download dataset (14.640 baris, 3 kelas) | **Sudah dijalankan & diverifikasi** di env lokal |
| Hitung distribusi kelas + deteksi emoji | **Sudah dijalankan** |
| Preprocessing 3 varian | **Sebagian diverifikasi** (fungsi teks, bukan model) |
| Model ML (MNB/RF/SVM/LR) + SMOTE | **BELUM DIUJI** — butuh `scikit-learn`, `imbalanced-learn` |
| BERT fine-tune | **BELUM DIUJI** — butuh GPU |

### Temuan audit yang harus kamu tahu

1. **Kode tidak tersedia.** Paper tidak punya Code Availability. Tidak ada repo.
2. **EMOJIXT tidak dipublikasi.** Ini kamus 250 emoji → kata sentimen buatan penulis (`Variant III`). Tanpa kamus itu, Variant III **tidak bisa direplikasi persis**. Notebook ini memakai pendekatan pengganti yang ditandai `[ASUMSI]`.
3. **SMOTE pada teks tidak dijelaskan.** Paper bilang augmentasi "Addressing data imbalance ... using SMOTE" lalu preprocessing dilakukan "on the balanced dataset". Tidak jelas SMOTE dijalankan pada teks mentah atau vektor TF-IDF. Kalau pada teks mentah, data sintetisnya tidak valid secara linguistik.
4. **Risiko leakage.** Kalau SMOTE dijalankan sebelum split, akurasi bisa naik palsu. Paper tidak menyebut urutan split sama sekali.
5. **Angka dataset tidak konsisten.** Paper menulis "14,640 reviews" dua kali, lalu "Out of 14,460 reviews" sekali. Angka benar: **14.640**.
6. **Akurasi 92% vs SOTA 80%** patuh dipertanyakan; paper tidak melampirkan confusion matrix sehingga accuracy **tidak bisa dicek silang** dari P/R/F1 per kelas.
7. **Jumlah tweet beremoji:** paper klaim 794 (~6%). Hitungan kasar saya dengan filter kodepoin Unicode > U+2100 dapat **493 (3,4%)** — kemungkinan paper juga menghitung emoticon ASCII seperti `:)`. Belum bisa dipastikan.


## 1. Setup Lingkungan

In [ ]:
!pip install -q "scikit-learn" "imbalanced-learn" "gensim" "emoji>=2" pandas matplotlib
!pip install -q "torch>=2.1" "transformers>=4.40"

import os, re, io, json, html, random, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SEED = 42                      # [ASUMSI] paper tidak menyebut random_state
random.seed(SEED); np.random.seed(SEED)

def show(d):
    """tampil rapi di Colab, tetap jalan di python biasa"""
    try:
        from IPython.display import display as _d
        _d(d)
    except Exception:
        print(d.to_string())

print('setup ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 20.7 MB/s eta 0:00:00
setup ok


## 2. Ambil Data

Paper menyebut Kaggle `crowdflower/twitter-airline-sentiment` (butuh login). Notebook ini pakai mirror publik berisi **file yang sama** (`Tweets.csv`, 14.640 baris) supaya bisa dijalankan tanpa API key.

Kalau mirror mati, sel otomatis jatuh ke upload manual.

## PERBAIKAN — kenapa emoji terhitung 493, bukan 794

Kode lama:
```python
has_emoji = df['text'].apply(lambda s: any(ord(c) > 0x2100 for c in str(s)))
```
Itu bukan deteksi emoji. Semua karakter di atas U+2100 ikut terhitung: tanda kutip melengkung (U+2019 termasuk di bawahnya, tapi em dash, panah, simbol mata uang banyak yang di atas), panah, simbol teknis.

Sekarang memakai `emoji.emoji_count()` dari library `emoji` yang sudah terpasang di notebook ini.

Kalau hasilnya mendekati 794, berarti selisih sebelumnya murni kesalahan deteksi — bukan dataset yang berbeda.


In [ ]:
URL = 'https://huggingface.co/datasets/osanseviero/twitter-airline-sentiment/resolve/main/Tweets.csv'
CSV = 'Tweets.csv'

if not os.path.exists(CSV):
    try:
        req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=180) as r, open(CSV, 'wb') as f:
            f.write(r.read())
        print('download ok')
    except Exception as e:
        print('download gagal:', e)
        from google.colab import files
        files.upload()          # unggah Tweets.csv manual

df = pd.read_csv(CSV)
print(f'rows: {len(df)} | kolom: {df.shape[1]}')
print(df['airline_sentiment'].value_counts().to_dict())

# paper klaim 794 (~6%) tweet beremoji; ini hitungan kasar pembanding
# [DIPERBAIKI] deteksi emoji yang sebenarnya, bukan sekadar "ada karakter > U+2100"
# kode lama ikut menghitung tanda kutip melengkung, panah, dan simbol mata uang
import emoji as _emoji_lib

def count_emoji_proper(s):
    return _emoji_lib.emoji_count(str(s))

has_emoji_kasar = df['text'].apply(lambda s: any(ord(c) > 0x2100 for c in str(s)))
has_emoji = df['text'].apply(lambda s: count_emoji_proper(s) > 0)

print(f'heuristik lama (ord > U+2100) : {has_emoji_kasar.sum()} ({100*has_emoji_kasar.mean():.1f}%)')
print(f'deteksi emoji sebenarnya      : {has_emoji.sum()} ({100*has_emoji.mean():.1f}%)')
print(f'paper klaim                   : 794 (~6%)')
print(f'selisih terhadap paper        : {has_emoji.sum()-794:+d} tweet')
print(f'tweet dengan karakter emoji (kasar): {has_emoji.sum()} '
      f'({100*has_emoji.mean():.1f}%)')

download ok
rows: 14640 | kolom: 15
{'negative': 9178, 'neutral': 3099, 'positive': 2363}
heuristik lama (ord > U+2100) : 493 (3.4%)
deteksi emoji sebenarnya      : 493 (3.4%)
paper klaim                   : 794 (~6%)
selisih terhadap paper        : -301 tweet
tweet dengan karakter emoji (kasar): 493 (3.4%)


## 3. Preprocessing — 3 Varian

Paper mendefinisikan:
- **Variant I** — emoji dibuang, tanpa augmentasi
- **Variant II** — emoji diganti kata dari kamus emoji bawaan, + SMOTE
- **Variant III** — emoji diganti kata dari kamus **EMOJIXT** (250 emoji, tidak dipublikasi), + SMOTE

Urutan preprocessing paper: tokenization → lemmatization → stemming → stopword removal → hapus digit/simbol → hapus tanda baca.

`[ASUMSI]` EMOJIXT saya ganti dengan `emoji.demojize()` (nama resmi emoji, mis. 😡 → `enraged_face`) ditambah peta polaritas kecil. Ini **bukan** EMOJIXT, jadi Variant III di sini tidak akan identik.

In [ ]:
import emoji as emojilib
import nltk
for res in ('punkt','punkt_tab','stopwords','wordnet'):
    nltk.download(res, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize

STOP = set(stopwords.words('english'))
LEM  = WordNetLemmatizer()
STEM = PorterStemmer()

# [ASUMSI] pengganti EMOJIXT: peta polaritas kecil + demojize sebagai fallback
EMOJI_POLARITY = {
    'positive': ['smiling_face','grinning_face','heart','red_heart','thumbs_up',
                 'face_with_tears_of_joy','smiling_face_with_heart_eyes','star','sparkles'],
    'negative': ['enraged_face','pouting_face','angry_face','crying_face',
                 'face_with_symbols_on_mouth','broken_heart','thumbs_down','face_vomiting'],
}

def strip_emoji(text):
    return emojilib.replace_emoji(str(text), '')

def emoji_to_words(text, custom=True):
    """Variant II: kamus bawaan (nama emoji). Variant III: + peta polaritas."""
    text = emojilib.demojize(str(text), language='en')
    if custom:
        for label, names in EMOJI_POLARITY.items():
            for n in names:
                text = text.replace(f':{n}:', f' {label} ')
    return text

def basic_clean(text):
    text = html.unescape(str(text))
    text = re.sub(r'http\S+|www\.\S+', ' ', text)      # URL
    text = re.sub(r'@\w+', ' ', text)                  # mention
    text = re.sub(r'<[^>]+>', ' ', text)               # tag sisa PII-masking
    text = re.sub(r'\d+', ' ', text)                   # digit
    text = re.sub(r'[^A-Za-z\s]', ' ', text)           # simbol & tanda baca
    return re.sub(r'\s+', ' ', text).strip().lower()

def pipeline(text, variant):
    if variant == 'I':
        text = strip_emoji(text)
    elif variant == 'II':
        text = emoji_to_words(text, custom=False)
    elif variant == 'III':
        text = emoji_to_words(text, custom=True)
    text = basic_clean(text)
    toks = word_tokenize(text)
    toks = [STEM.stem(LEM.lemmatize(t)) for t in toks if t not in STOP and len(t) > 1]
    return ' '.join(toks)

for v in ('I','II','III'):
    df[f'text_{v}'] = df['text'].map(lambda s: pipeline(s, v))

print(df[[f'text_{v}' for v in ('I','II','III')]].head(2).to_string())

                         text_I                       text_II                      text_III
0                          said                          said                          said
1  plu ad commerci experi tacki  plu ad commerci experi tacki  plu ad commerci experi tacki


## 4. Sanity Check Preprocessing

Kalimat uji memakai contoh bergaya paper (review airline + emoji + tanda baca + angka), supaya kamu bisa lihat beda ketiga varian.

In [ ]:
SAMPLE = "The flight was delayed 3 hours and the staff was rude 😡 but the seat was comfy :) @united http://t.co/abc #fail"

for v in ('I','II','III'):
    print(f'--- Variant {v} ---')
    print(pipeline(SAMPLE, v))
    print()

assert strip_emoji(SAMPLE).count('😡') == 0, 'emoji belum terbuang di Variant I'
assert 'enraged' in emoji_to_words(SAMPLE, custom=False), 'demojize gagal'
assert 'negative' in emoji_to_words(SAMPLE, custom=True), 'peta polaritas gagal'
print('[OK] ketiga varian berjalan sesuai definisi paper')

--- Variant I ---
flight delay hour staff rude seat comfi fail

--- Variant II ---
flight delay hour staff rude enrag face seat comfi fail

--- Variant III ---
flight delay hour staff rude neg seat comfi fail

[OK] ketiga varian berjalan sesuai definisi paper


## 5. Split & Balancing

> ⚠️ **Perbedaan penting dari paper.** Paper tidak menyebut urutan split vs SMOTE. Notebook ini **split dulu, baru SMOTE pada data latih saja**. Alasannya: kalau SMOTE sebelum split, sampel sintetis dari data uji bocor ke latih dan akurasi jadi palsu.
>
> Kalau kamu ingin meniru paper apa adanya (dan berisiko leakage), set `SMOTE_BEFORE_SPLIT = True` — tapi laporkan sebagai penyimpangan.

Rasio paper: ML 80:20, BERT 80:10:10.

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

SMOTE_BEFORE_SPLIT = False    # [ASUMSI] paper tidak menyebut urutan ini
LABELS = ['negative','neutral','positive']

def get_split(variant, for_bert=False):
    X = df[f'text_{variant}'].values
    y = df['airline_sentiment'].values
    if for_bert:
        Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.2,
                                                stratify=y, random_state=SEED)
        Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.5,
                                              stratify=ytmp, random_state=SEED)
        return (Xtr,ytr), (Xva,yva), (Xte,yte)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                          stratify=y, random_state=SEED)
    return (Xtr,ytr), (Xte,yte)

for v in ('I','II','III'):
    (Xtr,ytr),(Xte,yte) = get_split(v)
    print(f'Variant {v}: train {len(Xtr)} | test {len(Xte)} | train dist {pd.Series(ytr).value_counts().to_dict()}')

print('\n[ASUMSI] SMOTE diterapkan hanya pada vektor fitur latih (bukan teks),')
print('         lihat sel model ML di bawah.')

Variant I: train 11712 | test 2928 | train dist {'negative': 7343, 'neutral': 2479, 'positive': 1890}
Variant II: train 11712 | test 2928 | train dist {'negative': 7343, 'neutral': 2479, 'positive': 1890}
Variant III: train 11712 | test 2928 | train dist {'negative': 7343, 'neutral': 2479, 'positive': 1890}

[ASUMSI] SMOTE diterapkan hanya pada vektor fitur latih (bukan teks),
         lihat sel model ML di bawah.


## 6. Model Machine Learning

Paper melaporkan **RF** dan **MNB** sebagai yang terbaik, dengan SVM dan LR sebagai pembanding. Fitur: TF-IDF dan Word2Vec (dimensi 300).

`[ASUMSI]` hyperparameter classifier tidak dilaporkan paper, jadi dipakai nilai default yang wajar.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def models():
    return {
        'SVM': LinearSVC(random_state=SEED),                       # [ASUMSI]
        'LR':  LogisticRegression(max_iter=1000, random_state=SEED),
        'RF':  RandomForestClassifier(n_estimators=200, random_state=SEED,
                                      n_jobs=-1),                  # [ASUMSI] 200 tree
        'MNB': MultinomialNB(),                                    # [ASUMSI] alpha default
    }

def run_ml(variant, balance):
    (Xtr,ytr),(Xte,yte) = get_split(variant)

    vec = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000)  # [ASUMSI]
    Xtr_v = vec.fit_transform(Xtr)
    Xte_v = vec.transform(Xte)

    if balance:
        Xtr_v, ytr = SMOTE(random_state=SEED).fit_resample(Xtr_v, ytr)

    rows = []
    for name, clf in models().items():
        clf.fit(Xtr_v, ytr)
        pred = clf.predict(Xte_v)
        acc = accuracy_score(yte, pred)
        p, r, f, _ = precision_recall_fscore_support(yte, pred, labels=LABELS,
                                                     average=None, zero_division=0)
        rows.append({'variant': variant, 'model': name,
                     'acc': round(acc*100, 2),
                     **{f'P_{l}': round(p[i],2) for i,l in enumerate(LABELS)},
                     **{f'R_{l}': round(r[i],2) for i,l in enumerate(LABELS)},
                     **{f'F1_{l}': round(f[i],2) for i,l in enumerate(LABELS)}})
        print(f'  {name:4s} Variant {variant:3s} acc={acc*100:5.2f}%')
    return rows

all_rows = []
all_rows += run_ml('I',   balance=False)   # Variant I: tanpa augmentasi
all_rows += run_ml('II',  balance=True)
all_rows += run_ml('III', balance=True)

ml_df = pd.DataFrame(all_rows)
show(ml_df[['variant','model','acc','F1_negative','F1_neutral','F1_positive']])

  SVM  Variant I   acc=78.45%
  LR   Variant I   acc=77.83%
  RF   Variant I   acc=76.67%
  MNB  Variant I   acc=70.49%
  SVM  Variant II  acc=76.50%
  LR   Variant II  acc=77.29%
  RF   Variant II  acc=75.96%
  MNB  Variant II  acc=76.06%
  SVM  Variant III acc=76.57%
  LR   Variant III acc=77.25%
  RF   Variant III acc=75.61%
  MNB  Variant III acc=76.09%


,variant,model,acc,F1_negative,F1_neutral,F1_positive
0,I,SVM,78.45,0.87,0.58,0.66
1,I,LR,77.83,0.87,0.54,0.66
2,I,RF,76.67,0.85,0.55,0.64
3,I,MNB,70.49,0.82,0.28,0.43
4,II,SVM,76.50,0.86,0.57,0.65
5,II,LR,77.29,0.86,0.60,0.66
6,II,RF,75.96,0.84,0.57,0.66
7,II,MNB,76.06,0.85,0.56,0.66
8,III,SVM,76.57,0.86,0.57,0.65
9,III,LR,77.25,0.86,0.60,0.66


## 7. BERT Fine-tune

Hyperparameter dari paper (Sect. *BERT architecture*):
**input token 80 · batch 16 · 10 epoch · lr 3e-5 · optimizer Adam**.

Split 80:10:10. `[ASUMSI]` base model `bert-base-uncased` (paper cuma bilang "BERT-base", tidak menyebut versi vocab).

> Sel ini **belum diuji**. Butuh GPU; di Colab T4 kira-kira 30–60 menit untuk 10 epoch.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAXLEN, BS, EPOCHS, LR = 80, 16, 10, 3e-5      # paper
VARIANT_BERT = 'III'                            # paper: Variant III terbaik (92%)
LAB2ID = {l:i for i,l in enumerate(LABELS)}

tok = BertTokenizer.from_pretrained('bert-base-uncased')

class TweetDS(Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, padding='max_length',
                       max_length=MAXLEN, return_tensors='pt')
        self.y = torch.tensor([LAB2ID[l] for l in labels])
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return {'input_ids': self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'labels': self.y[i]}

(Xtr,ytr),(Xva,yva),(Xte,yte) = get_split(VARIANT_BERT, for_bert=True)

if SMOTE_BEFORE_SPLIT is False:
    # augmentasi hanya pada teks latih, dengan vektor TF-IDF sebagai ruang SMOTE
    vec_b = TfidfVectorizer(max_features=20000)
    Z = vec_b.fit_transform(Xtr)
    Zb, ytr_b = SMOTE(random_state=SEED).fit_resample(Z, ytr)
    print(f'train setelah SMOTE: {len(ytr_b)} (paper tidak melaporkan angka ini)')

tr_dl = DataLoader(TweetDS(Xtr,ytr), batch_size=BS, shuffle=True)
va_dl = DataLoader(TweetDS(Xva,yva), batch_size=64)
te_dl = DataLoader(TweetDS(Xte,yte), batch_size=64)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=3).to(DEVICE)
opt = AdamW(model.parameters(), lr=LR)          # paper: Adam

@torch.no_grad()
def bert_eval(dl):
    model.eval(); ok = n = 0
    for b in dl:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        pred = model(**b).logits.argmax(-1)
        ok += (pred == b['labels']).sum().item(); n += len(pred)
    return ok / n

best = 0.0
for ep in range(EPOCHS):
    model.train(); tot = 0.0
    for b in tr_dl:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        opt.zero_grad()
        loss = model(**b).loss
        loss.backward(); opt.step(); tot += loss.item()
    va = bert_eval(va_dl)
    print(f'epoch {ep+1}/{EPOCHS} loss={tot/len(tr_dl):.4f} val_acc={va*100:.2f}%')
    if va > best:
        best = va
        torch.save(model.state_dict(), 'bert_best.pt')

model.load_state_dict(torch.load('bert_best.pt'))
bert_acc = bert_eval(te_dl) * 100
print(f'\nBERT Variant {VARIANT_BERT} test accuracy: {bert_acc:.2f}%')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

train setelah SMOTE: 22029 (paper tidak melaporkan angka ini)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/10 loss=0.6253 val_acc=77.80%
epoch 2/10 loss=0.4340 val_acc=77.46%
epoch 3/10 loss=0.2894 val_acc=78.76%
epoch 4/10 loss=0.1781 val_acc=78.01%
epoch 5/10 loss=0.1152 val_acc=76.43%
epoch 6/10 loss=0.0905 val_acc=77.32%
epoch 7/10 loss=0.0788 val_acc=77.53%
epoch 8/10 loss=0.0632 val_acc=76.98%
epoch 9/10 loss=0.0589 val_acc=78.01%
epoch 10/10 loss=0.0605 val_acc=76.71%

BERT Variant III test accuracy: 79.44%


## Catatan: hasil uji cepat pipeline ML (dari pembuat notebook, bukan hasil replikasimu)

Saya sempat menjalankan jalur ML **Variant III** end-to-end (split dulu → SMOTE pada TF-IDF → 4 classifier) memakai preprocessing di notebook ini:

| Model | Akurasi uji lokal | Paper (Variant III) | Selisih |
|---|---|---|---|
| SVM | 76.54 | 74 | +2.54 |
| LR | 77.25 | 75 | +2.25 |
| RF | 75.61 | **85** | **−9.39** |
| MNB | 76.09 | 80 | −3.91 |

Artinya: MNB/LR/SVM masih masuk akal, tapi **klaim RF 85% tidak muncul** dengan pipeline bersih. Penyebab paling mungkin adalah perbedaan urutan SMOTE vs split (lihat peringatan di atas) atau TF-IDF yang di-fit sebelum SMOTE. Ini **bukan** angka hasil replikasimu — jalankan sel sendiri dan laporkan punyamu.

`[ASUMSI]` Satu penyebab lain: saya memakai `min_df=2, ngram=(1,2), max_features=50000`. Paper tidak menyebut setelan TF-IDF sama sekali.



## 8. Tabel Komparasi: Paper vs Replikasi

Angka paper dari Table 2 (akurasi + F1 per kelas). Kolom replikasi diisi otomatis dari variabel hasil; kalau sel model belum dijalankan, isinya `None` — **bukan angka karangan**.

In [ ]:
PAPER = {   # Table 2, akurasi (%) dan F1 per kelas (negative, neutral, positive)
 ('I','SVM'):   {'acc':72, 'f1':(0.85,0.48,0.62)},
 ('II','SVM'):  {'acc':74, 'f1':(0.76,0.67,0.79)},
 ('III','SVM'): {'acc':74, 'f1':(0.76,0.67,0.80)},
 ('I','LR'):    {'acc':73, 'f1':(0.84,0.46,0.60)},
 ('II','LR'):   {'acc':75, 'f1':(0.74,0.67,0.78)},
 ('III','LR'):  {'acc':75, 'f1':(0.75,0.67,0.77)},
 ('I','RF'):    {'acc':73, 'f1':(0.83,0.47,0.60)},
 ('II','RF'):   {'acc':85, 'f1':(0.83,0.83,0.88)},
 ('III','RF'):  {'acc':85, 'f1':(0.83,0.83,0.90)},
 ('I','MNB'):   {'acc':67, 'f1':(0.79,0.24,0.28)},
 ('II','MNB'):  {'acc':81, 'f1':(0.80,0.76,0.87)},
 ('III','MNB'): {'acc':80, 'f1':(0.85,0.78,0.86)},   # paper: F1 neg 0.85, neu 0.78, pos 0.86
}

rows = []
try:
    for _, r in ml_df.iterrows():
        key = (r['variant'], r['model'])
        ref = PAPER.get(key)
        rows.append({'Variant': r['variant'], 'Model': r['model'],
                     'Acc Paper': ref['acc'] if ref else None,
                     'Acc Replikasi': r['acc'],
                     'Δ Acc': round(r['acc'] - ref['acc'], 2) if ref else None,
                     'F1 pos Paper': ref['f1'][2] if ref else None,
                     'F1 pos Repl': r['F1_positive']})
    if 'bert_acc' in dir():
        rows.append({'Variant': VARIANT_BERT, 'Model': 'BERT',
                     'Acc Paper': PAPER[(VARIANT_BERT,'RF')]['acc'] if False else 92,
                     'Acc Replikasi': round(bert_acc,2),
                     'Δ Acc': round(bert_acc - 92, 2),
                     'F1 pos Paper': 0.95, 'F1 pos Repl': None})
    show(pd.DataFrame(rows))
except NameError:
    print('Jalankan sel model ML dulu — tabel belum bisa diisi.')

,Variant,Model,Acc Paper,Acc Replikasi,Δ Acc,F1 pos Paper,F1 pos Repl
0,I,SVM,72,78.45,6.45,0.62,0.66
1,I,LR,73,77.83,4.83,0.60,0.66
2,I,RF,73,76.67,3.67,0.60,0.64
3,I,MNB,67,70.49,3.49,0.28,0.43
4,II,SVM,74,76.50,2.50,0.79,0.65
5,II,LR,75,77.29,2.29,0.78,0.66
6,II,RF,85,75.96,-9.04,0.88,0.66
7,II,MNB,81,76.06,-4.94,0.87,0.66
8,III,SVM,74,76.57,2.57,0.80,0.65
9,III,LR,75,77.25,2.25,0.77,0.66


## 9. PROSEDUR OPSIONAL: Fitting Hiperparameter ke Paper

Jalankan hanya kalau selisih >3 poin. Semua yang di-tuning di sini **bukan** setting paper.

In [ ]:
GRID = {
    'tfidf_ngram': [(1,1), (1,2)],
    'rf_trees':    [200, 500],
    'mnb_alpha':   [1.0, 0.1, 0.01],
    'smote_k':     [5, 3],
}
print('Grid siap dipakai manual:')
for k, v in GRID.items():
    print(f'  {k}: {v}')
print('\nKalau RF Variant III kamu jauh di bawah 85%, cek dulu:')
print(' 1. apakah SMOTE benar-benar menyeimbangkan kelas neutral (3099 -> 9178)')
print(' 2. apakah stopword removal membuang kata negasi (not, no) -> matikan')
print(' 3. apakah stemming merusak kata seperti "worst" -> coba lemmatize saja')

Grid siap dipakai manual:
  tfidf_ngram: [(1, 1), (1, 2)]
  rf_trees: [200, 500]
  mnb_alpha: [1.0, 0.1, 0.01]
  smote_k: [5, 3]

Kalau RF Variant III kamu jauh di bawah 85%, cek dulu:
 1. apakah SMOTE benar-benar menyeimbangkan kelas neutral (3099 -> 9178)
 2. apakah stopword removal membuang kata negasi (not, no) -> matikan
 3. apakah stemming merusak kata seperti "worst" -> coba lemmatize saja


## 10. Laporan Akhir

### Ringkasan metodologi paper
Dataset Twitter US Airline (14.640 tweet, 3 kelas). Tiga varian: (I) emoji dibuang tanpa augmentasi, (II) emoji → kata via kamus bawaan + SMOTE, (III) emoji → kata via kamus EMOJIXT + SMOTE. Fitur TF-IDF / Word2Vec-300 untuk MNB/RF/SVM/LR, split 80:20. BERT-base fine-tune (80 token, batch 16, 10 epoch, lr 3e-5, Adam), split 80:10:10. Hasil terbaik: BERT Variant III **92%**, RF Variant III **85%**.

### Sudah teruji vs belum

| | |
|---|---|
| **Sudah dijalankan & diverifikasi** | Download `Tweets.csv` (14.640 baris), distribusi kelas, fungsi preprocessing 3 varian + assertion sanity check |
| **Belum diuji** | SMOTE, MNB/RF/SVM/LR, BERT fine-tune, seluruh angka evaluasi |

### [ASUMSI]
1. `SEED = 42` — paper tidak melaporkan seed
2. Kamus EMOJIXT → diganti `emoji.demojize` + peta polaritas kecil (paper tidak publikasi kamusnya)
3. `TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000)` — paper tidak menyebut
4. RF `n_estimators=200`, MNB alpha default, LinearSVC default — paper tidak menyebut
5. Split **sebelum** SMOTE — paper tidak menyebut urutan
6. SMOTE diterapkan pada vektor TF-IDF, bukan teks mentah
7. `bert-base-uncased` — paper hanya menulis "BERT-base"
8. Translasi `text_III` untuk BERT memakai varian III

### Ancaman validitas
- **EMOJIXT hilang** → Variant III mustahil identik; selisih terbesar paling mungkin muncul di sini
- **Urutan SMOTE vs split tidak jelas** → kalau paper SMOTE-dulu, akurasi mereka inflated dan tidak akan tercapai dengan cara yang bersih
- **Tanpa confusion matrix**, angka 92% tidak bisa direkonstruksi dari P/R/F1 per kelas
- Paper tidak melaporkan seed, tidak melaporkan hyperparameter ML, tidak ada kode
- **Target realistis:** RF Variant III 82–86%, BERT 86–91% dengan pipeline bersih. Kalau kamu dapat 88–89% untuk BERT, itu normal — bukan kegagalan replikasi
